# DOJ: Hiring A Lot of Attorneys — But Still Shrinking

**Repo:** [github.com/abigailhaddad/doj-lawyer-hiring](https://github.com/abigailhaddad/doj-lawyer-hiring)

Tracks three things for DOJ series 0905 (General Attorney): headcount, actual new hires, and job postings — before and after January 20, 2025.

## Data sources

**OPM EHRI** — [`impactproject/opm-ehri-data`](https://huggingface.co/datasets/impactproject/opm-ehri-data) on HuggingFace  
Monthly accessions (new hires) and monthly employment snapshots (headcount). Queried server-side via DuckDB `hf://` — no local download.

**USAJobs** — R2 parquet mirror at `pub-317c58882ec04f329b63842c1eb65b0c.r2.dev`  
Historical postings from [`abigailhaddad/usajobs_historical`](https://github.com/abigailhaddad/usajobs_historical), stored as annual parquet files. Queried via DuckDB — no download.

Both sources use OPM's four-digit occupational series codes. Series `0905` is General Attorney. Filtering both to DOJ + 0905 puts them on the same population.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

INAUGURATION = '2025-01-20'
CHART_START  = '2024-01-01'
R2 = 'https://pub-317c58882ec04f329b63842c1eb65b0c.r2.dev'
HF = 'hf://datasets/impactproject/opm-ehri-data'

conn = duckdb.connect()

## Helper: pick latest version per month

EHRI files exist in multiple versions (v1, v2, v3) as OPM revises data. We pick the highest version per month.

In [ ]:
def latest_version_urls(glob_pattern, year_min=2022):
    all_files = conn.execute(f"SELECT file FROM glob('{glob_pattern}')").df()['file'].tolist()
    best = {}
    for f in all_files:
        name = f.split('/')[-1]
        parts = name.replace('.parquet', '').split('_')
        yyyymm, ver = parts[1], int(parts[2][1:])
        if int(yyyymm[:4]) >= year_min:
            if yyyymm not in best or ver > best[yyyymm][1]:
                best[yyyymm] = (f, ver)
    return [v[0] for v in sorted(best.values(), key=lambda x: x[0])]

## Part 1: EHRI Accessions — Who Actually Got Hired

In [ ]:
acc_urls = latest_version_urls(f'{HF}/accessions/*.parquet')
url_list = ', '.join(f"'{u}'" for u in acc_urls)

acc_df = conn.execute(f"""
    SELECT 
        regexp_extract(filename, 'accessions_([0-9]{{6}})', 1) AS yyyymm,
        SUM(CAST(count AS INTEGER)) AS accessions
    FROM read_parquet([{url_list}], filename=true)
    WHERE agency ILIKE '%JUSTICE%'
      AND occupational_series_code = '0905'
    GROUP BY 1 ORDER BY 1
""").df()

acc_df['date'] = pd.to_datetime(acc_df['yyyymm'], format='%Y%m')
print(f"Accessions: {acc_df['date'].min().strftime('%Y-%m')} → {acc_df['date'].max().strftime('%Y-%m')}")
acc_df.tail(8)

## Part 2: EHRI Employment — Total Headcount

In [ ]:
emp_urls = latest_version_urls(f'{HF}/employment/*.parquet')
emp_url_list = ', '.join(f"'{u}'" for u in emp_urls)

emp_df = conn.execute(f"""
    SELECT 
        regexp_extract(filename, 'employment_([0-9]{{6}})', 1) AS yyyymm,
        SUM(CAST(count AS INTEGER)) AS headcount
    FROM read_parquet([{emp_url_list}], filename=true)
    WHERE agency = 'DEPARTMENT OF JUSTICE'
      AND occupational_series_code = '0905'
    GROUP BY 1 ORDER BY 1
""").df()

emp_df['date'] = pd.to_datetime(emp_df['yyyymm'], format='%Y%m')
emp_df['net_change'] = emp_df['headcount'].diff()

peak   = emp_df.loc[emp_df['headcount'].idxmax()]
latest = emp_df.iloc[-1]
loss   = int(peak.headcount - latest.headcount)
print(f"Peak:   {int(peak.headcount):,} ({peak.date.strftime('%b %Y')})")
print(f"Latest: {int(latest.headcount):,} ({latest.date.strftime('%b %Y')})")
print(f"Net loss: {loss:,} attorneys ({loss/peak.headcount*100:.0f}%)")
emp_df.tail(8)

## Part 3: USAJobs Postings

The `occupationalSeries` field can contain multiple codes (e.g. `0905 - Attorney; 0950 - Paralegal`), so we filter with `LIKE '%0905%'`.

In [ ]:
years     = list(range(2019, 2027))
hist_urls = [f"'{R2}/data/historical_jobs_{y}.parquet'" for y in years]
curr_urls = [f"'{R2}/data/current_jobs_{y}.parquet'"   for y in [2024, 2025, 2026]]

post_df = conn.execute(f"""
    SELECT 
        LEFT(openDate, 7) AS month,
        COUNT(DISTINCT usajobsControlNumber) AS postings
    FROM read_parquet([{', '.join(hist_urls + curr_urls)}])
    WHERE hiringDepartmentName = 'Department of Justice'
      AND occupationalSeries LIKE '%0905%'
      AND openDate IS NOT NULL
    GROUP BY 1 ORDER BY 1
""").df()

post_df['date'] = pd.to_datetime(post_df['month'] + '-01')
post_df = post_df[post_df['date'] >= '2022-01-01'].reset_index(drop=True)
print(f"Postings: {post_df['month'].min()} → {post_df['month'].max()}")
post_df.tail(8)

## Part 4: Combine and derive separations

Separations (departures) = accessions − net headcount change. If they hired 140 people and headcount fell by 50, then 190 people left.

In [ ]:
plot_df = pd.merge(emp_df[['date','headcount','net_change']], acc_df[['date','accessions']], on='date', how='inner')
plot_df['separations'] = plot_df['accessions'] - plot_df['net_change']
plot_df = plot_df[plot_df['date'] >= CHART_START].copy()
post_plot = post_df[post_df['date'] >= CHART_START].copy()
print(f"Chart period: {plot_df['date'].min().strftime('%b %Y')} → {plot_df['date'].max().strftime('%b %Y')}")

## Part 5: Chart

In [ ]:
C_HIRE = '#2E86AB'
C_SEP  = '#E84855'
C_TOT  = '#3D405B'
C_POST = '#6A4C93'
BG     = '#FAFAFA'
GRID   = '#E8E8E8'
FONT   = 'Helvetica Neue, Arial, sans-serif'

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['New Hires (Accessions)', 'Departures', 'Total Headcount', 'USAJobs Postings'],
    vertical_spacing=0.16,
    horizontal_spacing=0.10,
)

fig.add_trace(go.Bar(x=plot_df['date'], y=plot_df['accessions'],  marker_color=C_HIRE, showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=plot_df['date'], y=plot_df['separations'], marker_color=C_SEP,  showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(
    x=plot_df['date'], y=plot_df['headcount'], mode='lines',
    line=dict(color=C_TOT, width=2.5),
    fill='tozeroy', fillcolor='rgba(61,64,91,0.08)',
    showlegend=False), row=2, col=1)
fig.add_trace(go.Bar(x=post_plot['date'], y=post_plot['postings'], marker_color=C_POST, showlegend=False), row=2, col=2)

fig.add_vline(x=INAUGURATION, line=dict(color='#999', width=1.5, dash='dot'))

for ax_i in range(1, 5):
    xax = f'xaxis{ax_i}' if ax_i > 1 else 'xaxis'
    yax = f'yaxis{ax_i}' if ax_i > 1 else 'yaxis'
    fig.update_layout(**{
        xax: dict(showgrid=False, tickformat='%b %Y', dtick='M2', tickangle=-40,
                  tickfont=dict(size=9, family=FONT), zeroline=False),
        yax: dict(showgrid=True, gridcolor=GRID, zeroline=True, zerolinecolor='#ccc',
                  tickfont=dict(size=10, family=FONT)),
    })

fig.update_layout(
    yaxis3=dict(tickformat=',d'),
    title=dict(
        text='<b>DOJ: Hiring A Lot of Attorneys — But Still Shrinking</b><br>'
             '<span style="font-size:13px;color:#666">Series 0905 (General Attorney) · Jan 2024–Apr 2026</span>',
        font=dict(size=20, family=FONT, color='#1a1a2e'),
        x=0.5, xanchor='center',
    ),
    paper_bgcolor=BG, plot_bgcolor=BG,
    font=dict(family=FONT, size=12, color='#333'),
    margin=dict(t=130, b=80, l=70, r=40),
    height=700, width=1150,
)

for ann in fig.layout.annotations:
    ann.font.update(size=13, family=FONT, color='#333')

fig.add_annotation(
    x=0.5, y=-0.09, xref='paper', yref='paper',
    text='Sources: OPM EHRI (impactproject/opm-ehri-data, HuggingFace) · USAJobs (abigailhaddad/usajobs_historical) · github.com/abigailhaddad/doj-lawyer-hiring',
    showarrow=False, font=dict(size=9, color='#aaa', family=FONT), xanchor='center',
)

fig.show()
fig.write_image('doj_4panel.png', scale=2)
print('Saved: doj_4panel.png')

## What the data shows

**Repo:** [github.com/abigailhaddad/doj-lawyer-hiring](https://github.com/abigailhaddad/doj-lawyer-hiring)

**The workforce is dramatically smaller.** DOJ peaked at ~13,130 series-0905 attorneys in early 2024. As of April 2026 (latest data), there are ~10,259 — a loss of ~2,871 attorneys (~22%). October 2025 saw a single-month drop of ~700, consistent with a RIF.

**The early-2025 hiring freeze was real.** New hires averaged ~27/month in Feb–Jun 2025 vs. ~75/month for the same period in 2024.

**Hiring has recovered — but can't keep up with attrition.** By 2026, monthly postings are the highest in the full dataset. Accessions are back to normal levels. But headcount is still falling, because departures continue to outpace new hires.

## Notes

- EHRI data lags 1–3 months; the most recent months may be incomplete
- Some DOJ positions fill outside USAJobs (Schedule A, lateral transfers, honors program)
- Series 0905 spans entry-level to senior litigators across all DOJ divisions
- All EHRI data is anonymous aggregate counts — no individual-level information
- `separations` here is derived: `accessions − net_headcount_change`; it includes retirements, resignations, firings, and transfers out